<a href="https://colab.research.google.com/github/JPTorrents/fieldwork-brigcolage/blob/main/INESSS_cvs_publications_people_and_products_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Objectif

Parcourir le sitemap “publications” de l’INESSS et extraire, pour chaque page publication :

- métadonnées de base : titre, URL, date, année, description, Notice Santécom
- thèmes
- équipe projet, normalisée par rôle
- produits de connaissance et leurs liens, sans télécharger les fichiers

Sorties

Le notebook produit quatre fichiers CSV :

- `publications_staging.csv` : vue brute canonique au niveau page
- `publications.csv` : table principale, une ligne par publication
- `publication_people.csv` : table enfant, une ligne par personne et par rôle
- `products.csv` : table enfant, une ligne par produit ou ressource liée

Contraintes de collecte

- respect de `robots.txt`
- délai entre requêtes HTTP
- aucune requête vers les PDF ; seuls les liens sont collectés

In [ ]:
# -*- coding: utf-8 -*-
# Colab/CPython 3.10+
# Scraper INESSS publications
# Purpose:
#   1. collect one canonical staging record per publication page
#   2. normalize nested people and product data into relational CSV outputs
#
# Outputs:
#   - publications_staging.csv
#   - publications.csv
#   - publication_people.csv
#   - products.csv
#
# Policy:
#   - respect robots.txt
#   - do not download linked documents
#   - do not issue HTTP requests to PDF URLs

#!pip -q install bs4 lxml tqdm pandas

import re, time, json, os, csv, unicodedata
import urllib.parse as up
import xml.etree.ElementTree as ET
import requests
from urllib import robotparser
from bs4 import BeautifulSoup
from tqdm import tqdm
import pandas as pd

import hashlib
from urllib.parse import urlparse

In [ ]:
# --------- Paramètres ----------
SITEMAP_URL = "https://www.inesss.qc.ca/sitemap.xml?sitemap=publications&cHash=f84a5bd60bab2d0e50c1727c78508e77"
USER_AGENT = "kind-academic-research-bot/1.0 (+mailto:joel.pereztorrents@mcgill.ca)"
FROM_EMAIL = "joel.pereztorrents@mcgill.ca"

# Pour tester rapidement, mettre DRY_RUN_LIMIT à 10 puis remettre à None pour tout
DRY_RUN_LIMIT = None  # ex: 10
FAST_MODE = False     # si True => réduit delay à 0.5s pour tests rapides (à laisser False pour scraping final)

# Dossiers & fichiers de sortie (Colab: /content)
OUT_DIR = "./inesss_publications"
os.makedirs(OUT_DIR, exist_ok=True)

CSV_PUBLICATIONS = os.path.join(OUT_DIR, "publications.csv")
CSV_PUBLICATION_PEOPLE = os.path.join(OUT_DIR, "publication_people.csv")
CSV_PRODUCTS = os.path.join(OUT_DIR, "products.csv")
CSV_STAGING = os.path.join(OUT_DIR, "publications_staging.csv")

SEP = "\\"  # séparateur interne pour champs multi-entrées

In [ ]:
# ---------- Session ----------
session = requests.Session()
session.headers.update({"User-Agent": USER_AGENT, "From": FROM_EMAIL})

In [ ]:
# ---------- Utils texte ----------
def clean_text(s): return re.sub(r"\s+", " ", s or "").strip()

def norm(s: str) -> str:
    # normalisation pour comparaisons (sans accents, lower)
    s = s or ""
    s = unicodedata.normalize("NFKD", s)
    s = s.encode("ascii", "ignore").decode("ascii")
    return s.lower().strip()

def make_publication_id(url: str) -> str:
    """
    Build a deterministic publication identifier from the URL path.
    Falls back to a short MD5 hash if the path does not contain a usable slug.
    """
    path = urlparse(url).path.rstrip("/")
    slug = path.split("/")[-1] if path else ""
    if slug:
        return slug
    return hashlib.md5(url.encode("utf-8")).hexdigest()[:16]

In [ ]:
# ---------- robots.txt ----------
def load_robots_txt(base="https://www.inesss.qc.ca/robots.txt"):
    robotText = robotparser.RobotFileParser()
    try:
        res = session.get(base, timeout=20)
        res.raise_for_status()
        robotText.parse(res.text.splitlines())
    except Exception as e:
        print(f"[WARN] robots.txt non chargé ({e}). On applique un délai 5s et on n'utilise pas can_fetch().")
        robotText.allow_all = True
    return robotText

robotText = load_robots_txt()
CRAWL_DELAY = robotText.crawl_delay(USER_AGENT) if hasattr(robotText, "crawl_delay") else None
if CRAWL_DELAY is None:
    CRAWL_DELAY = 5.0
if FAST_MODE:
    CRAWL_DELAY = 0.5  # uniquement pour tests rapides

def polite_get(url, max_retries=3, timeout=30):
    # Interdiction stricte de FETCH des PDFs (Disallow /*.pdf$) -> on collecte seulement leurs href
    if re.search(r"\.pdf($|[\?])", url, re.I):   # re: Python's built-in re module supporting regular expressions ('import re' at the begining')
                                                 # The line if re.search(r"\.pdf($|[\?])", url, re.I): checks if the given url string contains the sequence ".pdf" followed by either the very end of the string or a question mark, ignoring case.
        raise RuntimeError("Attempt to fetch PDF blocked by robots policy.")
    if hasattr(robotText, "can_fetch") and not robotText.can_fetch(USER_AGENT, url):
        raise RuntimeError(f"Blocked by robots.txt: {url}")
    for attempt in range(1, max_retries+1):
        time.sleep(CRAWL_DELAY)
        try:
            res = session.get(url, timeout=timeout)
            if res.status_code in (429, 503, 502):
                raise requests.HTTPError(f"HTTP {res.status_code}")
            res.raise_for_status()
            return res
        except Exception as e:
            if attempt == max_retries:
                raise
            sleep = CRAWL_DELAY * (1.5 ** (attempt-1))
            print(f"[WARN] Retry {attempt}/{max_retries} after error: {e} -> sleep {sleep:.1f}s")
            time.sleep(sleep)




In [ ]:
# ---------- Sitemap ----------

def parse_sitemap(sitemap_url):
    """
    Parse a sitemap or sitemap index and return a deduplicated list of
    {"url": ..., "lastmod": ...} dictionaries.
    """
    # Supports both <urlset> and <sitemapindex> structures.
    xml = polite_get(sitemap_url).text                         #URL en string
    root = ET.fromstring(xml)                                  #cree l'element 'root' du module ET (ElementTree) a partir de l'URL
    ns = {"sm": "http://www.sitemaps.org/schemas/sitemap/0.9"} #??? This defines a dictionary for XML namespaces. Sitemaps often use namespaces, and providing this dictionary is necessary to correctly find elements within the XML using methods like findall. "sm" is a common prefix used for the sitemap namespace URI.
    urls = []                                                  #An empty list is initialized to store the extracted URLs and their associated data
    # Gère à la fois <urlset><url>... et <sitemapindex>
    url_elems = root.findall(".//sm:url", ns)                  #This line searches the entire XML tree (.//) for all elements named url that are within the sitemap namespace (sm:)
    if not url_elems:
        # peut-être un index de sitemaps
        sm_elems = root.findall(".//sm:sitemap/sm:loc", ns)
        for se in sm_elems:
            child = se.text.strip()
            try:
                urls.extend(parse_sitemap(child))
            except Exception as e:
                print(f"[WARN] Sub-sitemap error {child}: {e}")
        return urls
    for u in url_elems:
        loc = u.find("sm:loc", ns)
        if loc is None or not loc.text:
            continue
        lastmod = u.find("sm:lastmod", ns)
        urls.append({
            "url": loc.text.strip(),
            "lastmod": (lastmod.text.strip() if lastmod is not None and lastmod.text else None)
        })
    # Dedup & tri. To remove duplicate URLs from the urls list and keeps the order of the first occurrence.
    seen = set()
    uniq = []
    for d in urls:
        if d["url"] not in seen:
            seen.add(d["url"])
            uniq.append(d)
    return uniq #the list of unique URLs found in the sitemap(s), along with their last modification dates if available.

DATE_RE = re.compile(r"(\d{2})-(\d{2})-(\d{4})\s*\|\s*(.+)")

In [ ]:
# ---------- Recherche d'entêtes ----------

def find_header(soup, titles, levels=(1,2,3,4)):
    tags = soup.find_all([f"h{i}" for i in levels])
    tgt = [norm(t) for t in titles]
    for h in tags:
        if norm(h.get_text(" ", strip=True)) in tgt:
            return h
    return None

def find_header_contains(soup, needle, levels=(1,2,3,4,5,6)):
    target = norm(needle)
    for i in levels:
        for h in soup.find_all(f"h{i}"):
            if target in norm(h.get_text(" ", strip=True)):
                return h
    return None

def iter_until_next_header(start):
    # itère sur les siblings suivants jusqu'au prochain h1..h4
    for sib in start.find_all_next():
        if sib == start:
            continue
        if sib.name and re.match(r"h[1-4]$", sib.name, re.I):
            break
        yield sib

In [ ]:
# ---------- Parsing champs spécifiques ----------
DATE_ISO_RE   = re.compile(r"\b(19|20)\d{2}-(0[1-9]|1[0-2])-(0[1-9]|[12]\d|3[01])\b")  # YYYY-MM-DD
DATE_DDMM_RE  = re.compile(r"\b(0[1-9]|[12]\d|3[01])-(0[1-9]|1[0-2])-(19|20)\d{2}\b")   # DD-MM-YYYY


def parse_title_date_themes(soup):
    # Titre
    h1 = soup.find("h1")
    title = clean_text(h1.get_text(" ", strip=True)) if h1 else None

    date_iso, themes_join = None, None

    # 1) Chemin privilégié : <header class="single__header"> ... <h2 class="... single__subtitle"> <span>DATE</span> <span>|</span> <span>THEME</span>
    header = soup.find("header", class_=lambda c: c and "single__header" in c)
    if header:
        h2 = header.find("h2", class_=lambda c: c and "single__subtitle" in c)
        if h2:
            spans = [clean_text(s.get_text(" ", strip=True)) for s in h2.find_all("span")]
            spans = [s for s in spans if s]  # non vides

            # DATE = premier span qui matche YYYY-MM-DD ou DD-MM-YYYY
            for s in spans:
                if DATE_ISO_RE.fullmatch(s):
                    date_iso = s  # déjà en ISO
                    break
                m = DATE_DDMM_RE.fullmatch(s)
                if m:
                    dd, mm, yyyy = m.group(1), m.group(2), m.group(3)
                    date_iso = f"{yyyy}-{mm}-{dd}"
                    break

            # THEMES = spans non vides qui ne sont ni la date ni le séparateur "|"
            candidate_themes = []
            for s in spans:
                if s == "|" or DATE_ISO_RE.fullmatch(s) or DATE_DDMM_RE.fullmatch(s):
                    continue
                candidate_themes.append(s)
            # Parfois il n’y a qu’un seul thème ; parfois plusieurs (on garde l’ordre)
            if candidate_themes:
                # enlever séparateurs résiduels " | " s'il y en a
                candidate_themes = [t for t in candidate_themes if t.strip("|").strip()]
                themes_join = SEP.join(candidate_themes) if candidate_themes else None

    # 2) Fallbacks (ancienne logique) si on n’a pas trouvé
    if date_iso is None or themes_join is None:
        # tenter h2 suivant h1 avec motif "date | thème"
        h2_after = h1.find_next("h2") if h1 else None
        text_h2  = clean_text(h2_after.get_text(" ", strip=True)) if h2_after else ""
        # détecter la date dans cette ligne
        m_iso = DATE_ISO_RE.search(text_h2)
        m_dmY = DATE_DDMM_RE.search(text_h2)
        if date_iso is None:
            if m_iso:
                date_iso = m_iso.group(0)
            elif m_dmY:
                dd, mm, yyyy = m_dmY.group(1), m_dmY.group(2), m_dmY.group(3)
                date_iso = f"{yyyy}-{mm}-{dd}"
        # extraire thèmes à droite d'un '|' si présent
        if themes_join is None and "|" in text_h2:
            right = text_h2.split("|", 1)[1]
            parts = re.split(r"\s*[;,/·•]\s*", right)
            parts = [clean_text(x) for x in parts if clean_text(x)]
            themes_join = SEP.join(parts) if parts else None

    return title, date_iso, themes_join


def extract_santecom_info(soup):
    santecom_url, santecom_id = None, None
    for a in soup.find_all("a", href=True):
        href = a["href"].strip()
        if "santecom" in href.lower():
            santecom_url = href
            # Essayer d'attraper un numéro à proximité
            vicinity = a.parent.get_text(" ", strip=True) if a.parent else a.get_text(" ", strip=True)
            mnum = re.search(r"Notice\s+Sant[eé]com\s*:\s*([0-9]+)", vicinity, re.I)
            if mnum:
                santecom_id = mnum.group(1)
            else:
                # parfois juste "Santécom" + numéro à côté
                mnum2 = re.search(r"\b([0-9]{5,})\b", vicinity)
                if mnum2:
                    santecom_id = mnum2.group(1)
            break
    return santecom_id, santecom_url

def _is_small_font_div(el):
    if el.name != "div": return False
    style = el.get("style") or ""
    # gère ".8em" et "0.8em", espaces variés, casse indifférente
    return bool(re.search(r"font-size\s*:\s*0?\.8em", style, flags=re.I))

def extract_description_after_santecom(soup):
    # 1) Si lien Santécom présent, on part de là
    anchor = None
    for a in soup.find_all("a", href=True):
        if "santecom" in a["href"].lower():
            anchor = a
            break

def collect_between(start_node):
    chunks = []
    for el in start_node.find_all_next():
        if el == start_node:
            continue
        # Stop à la prochaine section (nouvel en-tête)
        if el.name and re.match(r"h[1-6]$", el.name, re.I):
            break
        # Paragraphes
        if el.name == "p":
            txt = clean_text(el.get_text(" ", strip=True))
            if txt: chunks.append(txt)
        # Listes
        if el.name in ("ul", "ol"):
            for li in el.find_all("li", recursive=False):
                txt = clean_text(li.get_text(" ", strip=True))
                if txt: chunks.append(txt)
        # Petits blocs d’appoint (font-size: .8em)
        if _is_small_font_div(el):
            txt = clean_text(el.get_text(" ", strip=True))
            if txt: chunks.append(txt)
    return chunks

    if anchor:
        desc = collect_between(anchor)
        if desc:
            return SEP.join(desc)

    # 2) Fallback : depuis le haut de page (h2, sinon h1), jusqu'à "Documentation" / "Équipe projet"
    start = soup.find("h2") or soup.find("h1") or soup
    chunks = []
    for el in start.find_all_next():
        if el.name and re.match(r"h[1-6]$", el.name, re.I):
            label = norm(el.get_text(" ", strip=True))
            if label in {"documentation", "equipe projet", "project team", "team"}:
                break
        if el.name == "p":
            txt = clean_text(el.get_text(" ", strip=True))
            if txt: chunks.append(txt)
        if el.name in ("ul", "ol"):
            for li in el.find_all("li", recursive=False):
                txt = clean_text(li.get_text(" ", strip=True))
                if txt: chunks.append(txt)
        if _is_small_font_div(el):
            txt = clean_text(el.get_text(" ", strip=True))
            if txt: chunks.append(txt)

    return SEP.join(chunks) if chunks else None

def split_multi(value, sep=SEP):
    if value is None:
        return []
    if isinstance(value, float) and pd.isna(value):
        return []
    return [v.strip() for v in str(value).split(sep) if v.strip()]



In [ ]:
# ---- Equipe projet → colonnes dédiées ----
# Normalisation des libellés de rôles -> colonnes cibles
ROLE_TO_COL = {
    # Auteurs
    "auteur": "Auteur(s)", "auteurs": "Auteur(s)", "author": "Auteur(s)",
    "authors": "Auteur(s)", "author(s)": "Auteur(s)", "redaction": "Auteur(s)",
    # Collaboration
    "collaboration": "Collaboration", "collaborateurs": "Collaboration",
    "collaborator": "Collaboration", "collaborators": "Collaboration",
    "contributeurs": "Collaboration", "contributors": "Collaboration",
    # Transfert de connaissances
    "transfert de connaissances": "Transfert de connaissances",
    "knowledge transfer": "Transfert de connaissances",
    # Coordination scientifique
    "coordination scientifique": "Coordination scientifique",
    "scientific coordination": "Coordination scientifique",
    # Direction scientifique
    "direction scientifique": "Direction scientifique",
    "scientific direction": "Direction scientifique",
    "scientific director": "Direction scientifique",
    # Révision scientifique
    "revision scientifique": "Révision scientifique",
    "révision scientifique": "Révision scientifique",
    "scientific review": "Révision scientifique",
    "peer review": "Révision scientifique",  # au cas où
}

TEAM_COLS = ["Auteur(s)", "Collaboration", "Transfert de connaissances",
             "Coordination scientifique", "Direction scientifique",
             "Révision scientifique"]

def extract_team_columns(soup):
    """
    Extract project-team information and normalize it into the predefined
    role columns used by the downstream long-format people table.

    Supported HTML patterns:
    - <dt>/<dd> definition lists
    - <p><strong>Role:</strong> ...</p>
    - role headings followed by <p> or <li> items
    """
    out = {c: [] for c in TEAM_COLS}

    def push(role_label, text_block):
        role_label_n = norm(role_label)
        col = None
        # mapping exact puis partiel
        col = ROLE_TO_COL.get(role_label_n)
        if not col:
            for k, v in ROLE_TO_COL.items():
                if k in role_label_n:
                    col = v; break
        if not col: return
        # découper par ; , • · \ ou " | " implicites
        parts = re.split(r"\s*[;,\u2022\u00B7\\]\s*", text_block)
        parts = [clean_text(p) for p in parts if clean_text(p)]
        # dédupe en conservant l’ordre
        seen, uniq = set(), []
        for p in parts:
            if p not in seen:
                seen.add(p); uniq.append(p)
        out[col].extend(uniq)

    # 1) <dl><dt>Role</dt><dd>Liste</dd></dl>
    for dt in soup.find_all("dt"):
        role = clean_text(dt.get_text(" ", strip=True))
        dd = dt.find_next("dd")
        if dd:
            text_block = clean_text(dd.get_text(" ", strip=True))
            if text_block:
                push(role, text_block)

    # 2) <p><strong>Role :</strong> Noms...</p>
    for strong in soup.find_all("strong"):
        label = clean_text(strong.get_text(" ", strip=True))
        # tolérer "Role :" / "Role -"
        if re.search(r":|-$", label):
            parent_text = clean_text(strong.parent.get_text(" ", strip=True)) if strong.parent else ""
            text_block = parent_text[len(label):].strip(" -:\u00A0")
            if text_block:
                push(label.rstrip(" :-"), text_block)

    # 3) Titres h2..h6 comme rôles, puis li/p jusqu’au prochain titre
    for i in range(2,7):
        for h in soup.find_all(f"h{i}"):
            role = clean_text(h.get_text(" ", strip=True))
            # on n'utilise pas les gros titres génériques
            if norm(role) in {"documentation","project team","equipe projet","equipe","team"}:
                continue
            bucket = []
            for sib in h.find_all_next():
                if sib == h: continue
                if sib.name and re.match(r"h[1-6]$", sib.name, re.I): break
                if sib.name in ("li","p"):
                    txt = clean_text(sib.get_text(" ", strip=True))
                    if txt: bucket.append(txt)
            if bucket:
                push(role, SEP.join(bucket))

    return {c: (SEP.join(v) if v else None) for c, v in out.items()}

ROLE_FIELD_MAP = {
    "authors_raw": "author",
    "collaboration_raw": "collaboration",
    "knowledge_transfer_raw": "knowledge_transfer",
    "scientific_coordination_raw": "scientific_coordination",
    "scientific_direction_raw": "scientific_direction",
    "scientific_review_raw": "scientific_review",
}

def build_publications_df(rows):  #explicit normalization functions for child tables
    pubs = []
    for row in rows:
        pubs.append({
            "publication_id": row["publication_id"],
            "ref_code": row["ref_code"],
            "url": row["url"],
            "title": row["title"],
            "publication_date": row["publication_date"],
            "year": row["year"],
            "description": row["description"],
            "notice_santecom": row["notice_santecom"],
            "notice_santecom_url": row["notice_santecom_url"],
            "themes_json": json.dumps(split_multi(row["themes_raw"])),
            "publication_type": row["publication_type"],
            "language": row["language"],
            "has_transfer_tools": row["has_transfer_tools"],
            "html_path": row["html_path"],
        })
    return pd.DataFrame(pubs).drop_duplicates(subset=["publication_id"])


def build_publication_people_df(rows):
    people_rows = []

    for row in rows:
        publication_id = row["publication_id"]

        for field, role in ROLE_FIELD_MAP.items():
            for position, person_name in enumerate(split_multi(row.get(field)), start=1):
                people_rows.append({
                    "publication_id": publication_id,
                    "role": role,
                    "person_name": person_name,
                    "position": position,
                })

    df_people = pd.DataFrame(people_rows)
    if not df_people.empty:
        df_people = df_people.drop_duplicates(subset=["publication_id", "role", "person_name"])
    return df_people


In [ ]:
# --- PRODUITS DE CONNAISSANCE: header explicite OU fallback "Documentation" ---

def extract_produits_connaissance(soup):
    items = []

    def collect_links(container):
        for a in container.find_all("a", href=True):
            href = up.urljoin("https://www.inesss.qc.ca/", a["href"].strip())
            label = clean_text(a.get_text(" ", strip=True))
            if label or href:
                items.append((label, href))

    h_pc = find_header_contains(soup, "produits de connaissance", levels=(1,2,3,4,5,6))
    if h_pc:
        for el in h_pc.find_all_next():
            if el == h_pc:
                continue
            if el.name and re.match(r"h[1-6]$", el.name, re.I):
                break
            if el.name in ("ul", "ol", "div", "p", "section"):
                collect_links(el)

    if not items:
        doc_h = find_header_contains(soup, "documentation", levels=(1,2,3,4,5,6))
        if doc_h:
            for el in doc_h.find_all_next():
                if el == doc_h:
                    continue
                if el.name and re.match(r"h[1-6]$", el.name, re.I):
                    break
                if el.name in ("ul", "ol", "div", "p", "section"):
                    collect_links(el)

    deduped = []
    seen = set()
    for label, href in items:
        key = (label, href)
        if key not in seen:
            seen.add(key)
            deduped.append({"label": label, "url": href})

    return deduped


#Building the products table
def infer_product_format(url: str) -> str | None:
    if not url:
        return None
    lower = url.lower()
    if ".pdf" in lower:
        return "pdf"
    if ".docx" in lower:
        return "docx"
    if ".doc" in lower:
        return "doc"
    if ".xlsx" in lower:
        return "xlsx"
    if ".zip" in lower:
        return "zip"
    return "link"

def build_products_df(rows):
    product_rows = []

    for row in rows:
        publication_id = row["publication_id"]
        items = json.loads(row.get("product_items_json") or "[]")

        for position, item in enumerate(items, start=1):
            label = (item.get("label") or "").strip() or None
            url = (item.get("url") or "").strip() or None

            product_rows.append({
                "publication_id": publication_id,
                "product_order": position,
                "product_title": label,
                "product_url": url,
                "product_format": infer_product_format(url),
            })

    df_products = pd.DataFrame(product_rows)
    if not df_products.empty:
        df_products = df_products.drop_duplicates(subset=["publication_id", "product_title", "product_url"])
    return df_products

In [ ]:
# ---------- Scraper principal ----------
def scrape_publication(url):
    res = polite_get(url)
    soup = BeautifulSoup(res.text, "lxml")

    titre, date_iso, themes = parse_title_date_themes(soup)
    santecom_id, santecom_url = extract_santecom_info(soup)
    description = extract_description_after_santecom(soup)
    team_cols = extract_team_columns(soup)
    product_items = extract_produits_connaissance(soup)


    publication_id = make_publication_id(url)
    year = int(date_iso[:4]) if date_iso else None

    row = {
        "publication_id": publication_id,
        "ref_code": None,
        "url": url,
        "title": titre,
        "publication_date": date_iso,
        "year": year,
        "description": description,
        "notice_santecom": santecom_id,
        "notice_santecom_url": santecom_url,
        "themes_raw": themes,
        "publication_type": "publication",
        "language": "fr",
        "has_transfer_tools": bool(team_cols.get("Transfert de connaissances")) or bool(product_items),
        "html_path": None,

        # raw nested fields kept for explosion
        "authors_raw": team_cols.get("Auteur(s)"),
        "collaboration_raw": team_cols.get("Collaboration"),
        "knowledge_transfer_raw": team_cols.get("Transfert de connaissances"),
        "scientific_coordination_raw": team_cols.get("Coordination scientifique"),
        "scientific_direction_raw": team_cols.get("Direction scientifique"),
        "scientific_review_raw": team_cols.get("Révision scientifique"),
        "product_items_json": json.dumps(product_items, ensure_ascii=False),
    }
    return row



In [ ]:
# Validation block & diagnosis
assert df_publications["publication_id"].is_unique, "publication_id must be unique in publications"
assert df_publications["url"].notna().all(), "url must be non-null in publications"

if not df_people.empty:
    assert set(df_people["publication_id"]).issubset(set(df_publications["publication_id"])), \
        "publication_people contains unknown publication_id values"

if not df_products.empty:
    assert set(df_products["publication_id"]).issubset(set(df_publications["publication_id"])), \
        "products contains unknown publication_id values"

print("Validation OK")

print(df_publications.isna().sum())
print(df_people["role"].value_counts(dropna=False))
print(df_products["product_format"].value_counts(dropna=False))

In [ ]:
# ---------- Execution ----------
print("Lecture du sitemap…")
entries = parse_sitemap(SITEMAP_URL)
urls = [d["url"] for d in entries]
print(f"URLs détectées: {len(urls)}")
if DRY_RUN_LIMIT:
    urls = urls[:DRY_RUN_LIMIT]

rows, errors = [], []
for url in tqdm(urls):
    try:
        rows.append(scrape_publication(url))
    except Exception as e:
        errors.append({"url": url, "error": str(e)})
        print(f"\n[ERROR] {url}: {e}")

df_staging = pd.DataFrame(rows)
df_publications = build_publications_df(rows)
df_people = build_publication_people_df(rows)
df_products = build_products_df(rows)

df_staging.to_csv(CSV_STAGING, index=False, quoting=csv.QUOTE_MINIMAL)
df_publications.to_csv(CSV_PUBLICATIONS, index=False, quoting=csv.QUOTE_MINIMAL)
df_people.to_csv(CSV_PUBLICATION_PEOPLE, index=False, quoting=csv.QUOTE_MINIMAL)
df_products.to_csv(CSV_PRODUCTS, index=False, quoting=csv.QUOTE_MINIMAL)

print(f"Staging: {CSV_STAGING}")
print(f"Publications: {CSV_PUBLICATIONS} ({len(df_publications)})")
print(f"Publication people: {CSV_PUBLICATION_PEOPLE} ({len(df_people)})")
print(f"Products: {CSV_PRODUCTS} ({len(df_products)})")
if errors:
    print(f"Erreurs: {len(errors)} (voir ci-dessous)")
    for e in errors[:5]:
        print(e)

Lecture du sitemap…
URLs détectées: 616


100%|██████████| 616/616 [55:04<00:00,  5.37s/it]


Fichier créé: /content/inesss_publications/inesss_publications_single.csv
